In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import json
from collections import defaultdict
import polars as pl
import duckdb

import xml.etree.ElementTree as ET
from collections import defaultdict

from datetime import datetime

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
data_dir = 'F:/Din/Study/Education/Projects/Thesis/data'

mimic_path = os.path.join(data_dir, 'mimic_iv')

hosp_path = os.path.join(mimic_path, 'hosp')

icu_path = os.path.join(mimic_path, 'icu')

note_path = os.path.join(mimic_path, 'note')

In [ ]:
sample_path = os.path.join(hosp_path, 'prescriptions.csv')

con = duckdb.connect()

query = f"""
SELECT * 
FROM read_csv_auto('{sample_path}',
ignore_errors = True
)
"""

result = con.execute(query)

# while True:
#     chunk = result.fetch_df_chunk(10000)  # number of rows per chunk
#     if chunk.empty:
#         break

#     print(chunk.shape)

chunk = result.fetch_df_chunk(1000)

#### Check table

Key Identifier:

- subject_id -> patient
- hadm_id -> hospital admission
- stay_id -> ICU stay
- note_id -> Patient note

### Note

#### note/radiology

In [7]:
chunk

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-RR-14,10000032,22595853,RR,14,2180-05-06 21:19:00,2180-05-06 23:32:00,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...
1,10000032-RR-15,10000032,22595853,RR,15,2180-05-06 23:00:00,2180-05-06 23:26:00,EXAMINATION: LIVER OR GALLBLADDER US (SINGLE ...
2,10000032-RR-16,10000032,22595853,RR,16,2180-05-07 09:55:00,2180-05-07 11:15:00,"INDICATION: ___ HCV cirrhosis c/b ascites, hi..."
3,10000032-RR-18,10000032,<NA>,RR,18,2180-06-03 12:46:00,2180-06-03 14:01:00,EXAMINATION: Ultrasound-guided paracentesis.\...
4,10000032-RR-20,10000032,<NA>,RR,20,2180-07-08 13:18:00,2180-07-08 14:15:00,EXAMINATION: Paracentesis\n\nINDICATION: ___...
...,...,...,...,...,...,...,...,...
1621182,16988043-RR-226,16988043,<NA>,RR,226,2169-02-19 21:18:00,2169-02-19 22:20:00,CHEST RADIOGRAPH PERFORMED ON ___.\n\n___.\n\n...
1621183,16988043-RR-227,16988043,22462961,RR,227,2169-02-23 16:21:00,2169-02-23 17:08:00,"CHEST, TWO VIEWS: ___ \n\nHISTORY: ___ femal..."
1621184,16988043-RR-228,16988043,22462961,RR,228,2169-02-23 22:18:00,2169-02-24 00:38:00,"INDICATION: Dyspnea and elevated D-dimer, on ..."
1621185,16988043-RR-229,16988043,22462961,RR,229,2169-02-25 10:39:00,2169-02-25 12:07:00,HISTORY: History of three failed renal transp...


In [10]:
print(chunk['text'].iloc[2])

INDICATION:  ___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD,
bioplar, PTSD, presented from OSH ED with worsening abd distension over past
week.  // SBP

TECHNIQUE:  Ultrasound guided diagnostic and therapeutic paracentesis

COMPARISON:  Abdominal ultrasound ___

FINDINGS: 

Limited grayscale ultrasound imaging of the abdomen demonstrated
moderateascites. A suitable target in the deepest pocket in the right lower
quadrant was selected for paracentesis.

PROCEDURE:  The procedure, risks, benefits and alternatives were discussed
with the patient and written informed consent was obtained.

A preprocedure time-out was performed discussing the planned procedure,
confirming the patient's identity with 3 identifiers, and reviewing a
checklist per ___ protocol.

Under ultrasound guidance, an entrance site was selected and the skin was
prepped and draped in the usual sterile fashion. 1% lidocaine was instilled
for local anesthesia.

A 5 ___ catheter was advanced into the largest fluid

#### note/radiology_detail

In [5]:
chunk

,note_id,subject_id,field_name,field_value,field_ordinal
0,10000032-RR-14,10000032,exam_code,C11,1
1,10000032-RR-14,10000032,exam_name,CHEST (PA & LAT),1
2,10000032-RR-15,10000032,exam_code,U314,1
3,10000032-RR-15,10000032,exam_code,U644,3
4,10000032-RR-15,10000032,exam_code,W82,2
...,...,...,...,...,...
2026779,13349017-RR-21,13349017,exam_name,CTA HEAD W&W/O C & RECONS,1
2026780,13349017-RR-21,13349017,exam_name,CTA NECK W&W/OC & RECONS,2
2026781,13349017-RR-22,13349017,cpt_code,36216,1
2026782,13349017-RR-22,13349017,cpt_code,75665,2


#### note/discharge

In [4]:
chunk

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25,2180-07-25 21:42:00,\nName: ___ Unit No: _...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07,2180-08-10 05:43:00,\nName: ___ Unit No: _...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25,2160-11-25 15:09:00,\nName: ___ Unit No: __...
...,...,...,...,...,...,...,...,...
331788,19999828-DS-6,19999828,29734428,DS,6,2147-08-04,2147-08-12 15:36:00,\nName: ___ Unit No: ___...
331789,19999828-DS-7,19999828,25744818,DS,7,2149-01-18,2149-01-19 07:03:00,\nName: ___ Unit No: ___...
331790,19999840-DS-20,19999840,26071774,DS,20,2164-07-28,2164-07-29 14:52:00,\nName: ___ Unit No: ___\...
331791,19999840-DS-21,19999840,21033226,DS,21,2164-09-17,2164-09-18 01:36:00,\nName: ___ Unit No: ___\...


In [5]:
print(chunk['text'].iloc[0])

 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergies / Adverse Drug Reactions
 
Attending: ___
 
Chief Complaint:
Worsening ABD distension and pain 
 
Major Surgical or Invasive Procedure:
Paracentesis

 
History of Present Illness:
___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED with worsening abd 
distension over past week.  
Pt reports self-discontinuing lasix and spirnolactone ___ weeks 
ago, because she feels like "they don't do anything" and that 
she "doesn't want to put more chemicals in her." She does not 
follow Na-restricted diets. In the past week, she notes that she 
has been having worsening abd distension and discomfort. She 
denies ___ edema, or SOB, or orthopnea. She denies f/c/n/v, d/c, 
dysuria. She had food poisoning a week ago from eating stale 
cake (n/v 20 min after fo

#### note/discharge_detail

In [7]:
chunk

,note_id,subject_id,field_name,field_value,field_ordinal
0,10000032-DS-21,10000032,author,___,1
1,10000032-DS-22,10000032,author,___,1
2,10000032-DS-23,10000032,author,___,1
3,10000032-DS-24,10000032,author,___,1
4,10000084-DS-17,10000084,author,___,1
...,...,...,...,...,...
186133,15614172-DS-13,15614172,author,___,1
186134,15614172-DS-14,15614172,author,___,1
186135,15614172-DS-15,15614172,author,___,1
186136,15614172-DS-16,15614172,author,___,1


In [8]:
chunk['field_value'].value_counts()

field_value
___    186138
Name: count, dtype: int64

### ICU

#### icu/procedureevents

In [26]:
chunk.head()

,subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,value,valueuom,...,orderid,linkorderid,ordercategoryname,ordercategorydescription,patientweight,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
0,10000032,29079034,39553978,18704,2180-07-23 14:43:00,2180-07-23 14:44:00,2180-07-23 14:43:00.000,225966,1.0,None,...,3050203,3050203,Procedures,Task,39.4,0,0,FinishedRunning,1.0,0
1,10000032,29079034,39553978,<NA>,2180-07-23 14:24:00,2180-07-23 23:50:00,2180-07-23 23:50:49.983,224275,566.0,min,...,769190,769190,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1
2,10000032,29079034,39553978,<NA>,2180-07-23 14:24:00,2180-07-23 23:50:00,2180-07-23 23:50:49.983,224277,566.0,min,...,2668055,2668055,Peripheral Lines,ContinuousProcess,39.4,1,0,FinishedRunning,566.0,1
3,10000690,25860671,37081114,17393,2150-11-02 20:00:00,2150-11-03 13:00:00,2150-11-03 14:50:00.000,224275,1020.0,min,...,301419,301419,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,1020.0,1
4,10000690,25860671,37081114,17393,2150-11-02 20:38:00,2150-11-03 11:59:00,2150-11-03 12:34:00.000,224277,921.0,min,...,5207974,5207974,Peripheral Lines,ContinuousProcess,55.3,1,0,FinishedRunning,921.0,1


#### icu/outputevents

In [24]:
chunk.head()

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valueuom
0,10000032,29079034,39553978,18704,2180-07-23 15:00:00,2180-07-23 16:00:00,226560,175.0,mL
1,10000690,25860671,37081114,8787,2150-11-06 08:00:00,2150-11-06 09:08:00,226559,15.0,mL
2,10000690,25860671,37081114,8787,2150-11-06 09:00:00,2150-11-06 09:08:00,226559,15.0,mL
3,10000690,25860671,37081114,8787,2150-11-06 10:00:00,2150-11-06 09:50:00,226559,12.0,mL
4,10000690,25860671,37081114,8787,2150-11-06 11:00:00,2150-11-06 13:42:00,226559,15.0,mL


#### icu/inputevents

In [20]:
chunk.head()

,subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,amount,amountuom,...,ordercomponenttypedescription,ordercategorydescription,patientweight,totalamount,totalamountuom,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
0,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:01:00,2180-07-23 18:56:00,226452,200.000000,mL,...,Main order parameter,Bolus,39.4,200.0,mL,0,0,FinishedRunning,200.0,200.0
1,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:30:00,2180-07-23 17:02:00,220862,49.999999,mL,...,Main order parameter,Continuous IV,39.4,50.0,mL,0,0,FinishedRunning,50.0,100.0
2,10000032,29079034,39553978,18704,2180-07-23 17:33:00,2180-07-23 18:03:00,2180-07-23 18:16:00,220862,49.999999,mL,...,Main order parameter,Continuous IV,39.4,50.0,mL,0,0,FinishedRunning,50.0,100.0
3,10000032,29079034,39553978,18704,2180-07-23 18:56:00,2180-07-23 18:57:00,2180-07-23 18:56:00,226452,100.000000,mL,...,Main order parameter,Bolus,39.4,100.0,mL,0,0,FinishedRunning,100.0,100.0
4,10000032,29079034,39553978,20925,2180-07-23 21:10:00,2180-07-23 21:11:00,2180-07-23 21:10:00,226452,100.000000,mL,...,Main order parameter,Bolus,39.4,100.0,mL,0,0,FinishedRunning,100.0,100.0


#### icu/ingredientevents

In [17]:
chunk

,subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,amount,amountuom,rate,rateuom,orderid,linkorderid,statusdescription,originalamount,originalrate
0,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:01:00,2180-07-23 18:56:00,220490,200.000000,mL,NaN,None,3782233,3782233,FinishedRunning,0,200.000000
1,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:01:00,2180-07-23 18:56:00,227075,200.000000,mL,NaN,None,3782233,3782233,FinishedRunning,0,200.000000
2,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:30:00,2180-07-23 17:02:00,220490,49.999999,mL,100.000000,mL/hour,9796595,9796595,FinishedRunning,0,50.000000
3,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:30:00,2180-07-23 17:02:00,226509,49.999999,mL,100.000000,mL/hour,9796595,9796595,FinishedRunning,0,50.000000
4,10000032,29079034,39553978,18704,2180-07-23 17:33:00,2180-07-23 18:03:00,2180-07-23 18:16:00,220490,49.999999,mL,100.000000,mL/hour,6689854,6689854,FinishedRunning,0,50.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2004074,11390058,27321750,37294629,14454,2130-10-20 23:01:00,2130-10-21 02:38:00,2130-10-21 02:44:00,227074,723.999997,mL,200.184326,mL/hour,5333093,5333093,FinishedRunning,0,724.000000
2004075,11390058,27321750,37294629,14454,2130-10-20 23:03:00,2130-10-21 01:13:00,2130-10-21 01:55:00,220490,65.300003,mL,30.138462,mL/hour,2343264,2343264,FinishedRunning,0,65.300003
2004076,11390058,27321750,37294629,14454,2130-10-20 23:03:00,2130-10-21 01:13:00,2130-10-21 01:55:00,227074,65.300003,mL,30.138462,mL/hour,2343264,2343264,FinishedRunning,0,65.300003
2004077,11390058,27321750,37294629,14454,2130-10-20 23:05:00,2130-10-20 23:06:00,2130-10-20 23:05:00,220490,100.000000,mL,NaN,None,5508035,5508035,FinishedRunning,0,100.000000


#### icu/icustay (icu stay info)

In [15]:
chunk

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113
...,...,...,...,...,...,...,...,...
94453,19999442,26785317,32336619,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370
94454,19999625,25304202,31070865,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741
94455,19999828,25744818,36075953,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995
94456,19999840,21033226,38978960,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766


#### icu/datetimeevents

In [12]:
chunk

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valueuom,warning
0,10000032,29079034,39553978,18704,2180-07-23 14:24:00,2180-07-23 14:24:00,225754,2180-07-23 14:24:00,Date,0
1,10000032,29079034,39553978,18704,2180-07-23 14:24:00,2180-07-23 14:24:00,225755,2180-07-23 14:24:00,Date,0
2,10000032,29079034,39553978,20925,2180-07-23 21:02:00,2180-07-23 21:02:00,225754,2180-07-23 00:00:00,Date,0
3,10000032,29079034,39553978,20925,2180-07-23 21:02:00,2180-07-23 21:02:00,225755,2180-07-23 00:00:00,Date,0
4,10000690,25860671,37081114,17393,2150-11-02 20:00:00,2150-11-03 04:53:00,225754,2150-11-02 04:53:00,Date,0
...,...,...,...,...,...,...,...,...,...,...
2030764,12017899,21273211,33969290,39595,2188-08-02 08:00:00,2188-08-02 10:16:00,224288,2188-07-28 00:00:00,Date,0
2030765,12017899,21273211,33969290,39595,2188-08-02 08:00:00,2188-08-02 10:16:00,224290,2188-08-01 00:00:00,Date,0
2030766,12017899,21273211,33969290,39595,2188-08-02 08:00:00,2188-08-02 10:19:00,224261,2188-08-01 00:00:00,Date,0
2030767,12017899,21273211,33969290,39595,2188-08-02 08:00:00,2188-08-02 10:19:00,224279,2188-08-01 23:00:00,Date and Time,0


#### icu/d_items

In [10]:
chunk

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,None,Text,<NA>,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,None,Date and time,<NA>,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,<NA>,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,<NA>,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,<NA>,NaN
...,...,...,...,...,...,...,...,...,...
4090,230172,Patient Reversed,Patient Reversed,procedureevents,3-Significant Events,None,Processes,<NA>,NaN
4091,230173,Patient - Fast Track Protocol,Patient - Fast Track Protocol,procedureevents,3-Significant Events,None,Processes,<NA>,NaN
4092,230174,Nerve block in OR,Nerve block in OR,procedureevents,3-Significant Events,None,Processes,<NA>,NaN
4093,230176,IUC Stabilization Device,IUC Stabilization Device,chartevents,GI/GU,None,Checkbox,<NA>,NaN


#### icu/chartevents

In [4]:
chunk

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226512,39.4,39.4,kg,0
1,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226707,60,60.0,Inch,0
2,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226730,152,152.0,cm,0
3,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,220048,SR (Sinus Rhythm),NaN,None,0
4,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,224642,Oral,NaN,None,0
...,...,...,...,...,...,...,...,...,...,...,...
20214509,10475619,27932862,38522215,27242,2185-12-04 00:00:00,2185-12-04 00:38:00,228407,No response,NaN,None,0
20214510,10475619,27932862,38522215,27242,2185-12-04 00:00:00,2185-12-04 00:38:00,228415,Absent,NaN,None,0
20214511,10475619,27932862,38522215,27242,2185-12-04 00:00:00,2185-12-04 00:38:00,228416,Absent,NaN,None,0
20214512,10475619,27932862,38522215,27242,2185-12-04 00:00:00,2185-12-04 00:40:00,223986,Diminished,NaN,None,0


In [7]:
agg = chunk.groupby(['subject_id', 'hadm_id', 'stay_id', 'charttime']).agg({'itemid':'count', 'warning':'count', 'valuenum':'mean'})

In [8]:
agg

itemid  warning    valuenum
subject_id hadm_id  stay_id  charttime                                       
10000032   29079034 39553978 2180-07-23 12:36:00       3        3   83.800000
                             2180-07-23 13:06:00       1        1    5.800000
                             2180-07-23 14:00:00       4        4   98.700000
                             2180-07-23 14:11:00       3        3   62.666667
                             2180-07-23 14:12:00       2        2   57.500000
...                                                  ...      ...         ...
10475619   27932862 38522215 2185-12-03 21:00:00      10       10  163.857143
                             2185-12-03 22:00:00      33       33  216.217391
                             2185-12-03 23:00:00      13       13  133.250000
                             2185-12-04 00:00:00      59       59  161.403448
                             2185-12-05 21:00:00       7        7   99.666667

[1055361 rows x 3 columns]

### Hosp

#### hosp/services

In [39]:
chunk.head()

,subject_id,hadm_id,transfertime,prev_service,curr_service
0,10000032,22595853,2180-05-06 22:24:57,None,MED
1,10000032,22841357,2180-06-26 18:28:08,None,MED
2,10000032,25742920,2180-08-05 23:44:50,None,MED
3,10000032,29079034,2180-07-23 12:36:04,None,MED
4,10000068,25022803,2160-03-03 23:17:17,None,MED


#### hosp/provider

In [37]:
chunk.head()

,provider_id
0,P00019
1,P0001J
2,P0001P
3,P000FQ
4,P000H1


#### hosp/procedures_icd (procedure for each admission)

In [35]:
chunk.head()

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
0,10000032,22595853,1,2180-05-07,5491,9
1,10000032,22841357,1,2180-06-27,5491,9
2,10000032,25742920,1,2180-08-06,5491,9
3,10000068,25022803,1,2160-03-03,8938,9
4,10000117,27988844,1,2183-09-19,0QS734Z,10


#### hosp/prescriptions (medications)

In [4]:
chunk.head()

,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,...,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10000032,22595853,12775705,10000032-55,55,P85UQ1,2180-05-08 08:00:00,2180-05-07 22:00:00,MAIN,Furosemide,...,008209,51079007320,40mg Tablet,None,40,mg,1,TAB,1,PO/NG
1,10000032,22595853,18415984,10000032-42,42,P23SJA,2180-05-07 02:00:00,2180-05-07 22:00:00,MAIN,Ipratropium Bromide Neb,...,021700,00487980125,2.5mL Vial,None,1,NEB,1,VIAL,4,IH
2,10000032,22595853,23637373,10000032-35,35,P23SJA,2180-05-07 01:00:00,2180-05-07 09:00:00,MAIN,Furosemide,...,008208,51079007220,20mg Tablet,None,20,mg,1,TAB,1,PO/NG
3,10000032,22595853,26862314,10000032-41,41,P23SJA,2180-05-07 01:00:00,2180-05-07 01:00:00,MAIN,Potassium Chloride,...,001275,00245004101,10mEq ER Tablet,None,40,mEq,4,TAB,1,PO
4,10000032,22595853,30740602,10000032-27,27,P23SJA,2180-05-07 00:00:00,2180-05-07 22:00:00,MAIN,Sodium Chloride 0.9% Flush,...,None,0,10 mL Syringe,None,3,mL,0.3,SYR,3,IV


#### hosp/poe_detail

In [31]:
chunk.head()

,poe_id,poe_seq,subject_id,field_name,field_value
0,10000032-101,101,10000032,Code status,Resuscitate (Full code)
1,10000032-109,109,10000032,Admit category,Admit to inpatient
2,10000032-109,109,10000032,Admit to,Medicine
3,10000032-114,114,10000032,Admit category,Admit to inpatient
4,10000032-114,114,10000032,Discharge Planning,Finalized


#### hosp/poe

In [29]:
chunk.head()

,poe_id,poe_seq,subject_id,hadm_id,ordertime,order_type,order_subtype,transaction_type,discontinue_of_poe_id,discontinued_by_poe_id,order_provider_id,order_status
0,10000032-100,100,10000032,22841357,2180-06-26 22:09:02,Medications,None,New,None,None,P992IN,Inactive
1,10000032-101,101,10000032,22841357,2180-06-26 22:09:02,General Care,Code status,New,None,None,P992IN,Inactive
2,10000032-102,102,10000032,22841357,2180-06-26 22:09:02,Nutrition,Nutrition consult,New,None,None,P992IN,Inactive
3,10000032-103,103,10000032,22841357,2180-06-26 22:09:02,Blood Bank,Blood tests,New,None,None,P992IN,Inactive
4,10000032-104,104,10000032,22841357,2180-06-26 22:09:02,Lab,None,New,None,None,P992IN,Inactive


#### hosp/pharmacy

In [27]:
chunk.head()

,subject_id,hadm_id,pharmacy_id,poe_id,starttime,stoptime,medication,proc_type,status,entertime,...,basal_rate,one_hr_max,doses_per_24_hrs,duration,duration_interval,expiration_value,expiration_unit,expirationdate,dispensation,fill_quantity
0,10000032,22595853,12775705,10000032-55,2180-05-08 08:00:00,2180-05-07 22:00:00,Furosemide,Unit Dose,Discontinued via patient discharge,2180-05-07 09:32:35,...,NaN,NaN,1,<NA>,Ongoing,36,Hours,NaT,Omnicell,<NA>
1,10000032,22595853,18415984,10000032-42,2180-05-07 02:00:00,2180-05-07 22:00:00,Ipratropium Bromide Neb,Unit Dose,Discontinued via patient discharge,2180-05-07 01:49:23,...,NaN,NaN,4,<NA>,Ongoing,36,Hours,NaT,Omnicell,<NA>
2,10000032,22595853,23637373,10000032-35,2180-05-07 01:00:00,2180-05-07 09:00:00,Furosemide,Unit Dose,Inactive (Due to a change order),2180-05-07 00:09:24,...,NaN,NaN,1,<NA>,Ongoing,36,Hours,NaT,Omnicell,<NA>
3,10000032,22595853,26862314,10000032-41,2180-05-07 01:00:00,2180-05-07 01:00:00,Potassium Chloride,Unit Dose,Discontinued,2180-05-07 00:09:24,...,NaN,NaN,1,1,Doses,36,Hours,NaT,Omnicell,<NA>
4,10000032,22595853,30740602,10000032-27,2180-05-07 00:00:00,2180-05-07 22:00:00,Sodium Chloride 0.9% Flush,Unit Dose,Discontinued via patient discharge,2180-05-07 00:00:54,...,NaN,NaN,3,<NA>,Ongoing,36,Hours,NaT,Floor Stock Item,<NA>


#### hosp/omr

In [24]:
chunk.head()

,subject_id,chartdate,seq_num,result_name,result_value
0,10000032,2180-04-27,1,Blood Pressure,110/65
1,10000032,2180-04-27,1,Weight (Lbs),94
2,10000032,2180-05-07,1,BMI (kg/m2),18.0
3,10000032,2180-05-07,1,Height (Inches),60
4,10000032,2180-05-07,1,Weight (Lbs),92.15


#### hosp/microbiologyevents

In [22]:
chunk.head()

,microevent_id,subject_id,hadm_id,micro_specimen_id,order_provider_id,chartdate,charttime,spec_itemid,spec_type_desc,test_seq,...,org_name,isolate_num,quantity,ab_itemid,ab_name,dilution_text,dilution_comparison,dilution_value,interpretation,comments
0,1,10000032,<NA>,1304715,P69FQC,2180-03-23,2180-03-23 11:51:00,70046,IMMUNOLOGY,1,...,None,<NA>,None,<NA>,None,None,None,NaN,None,___
1,2,10000032,<NA>,3342526,P69FQC,2180-03-23,2180-03-23 11:51:00,70093,Blood (Toxo),1,...,None,<NA>,None,<NA>,None,None,None,NaN,None,NEGATIVE FOR TOXOPLASMA IgG ANTIBODY BY EIA. ...
2,3,10000032,<NA>,3910370,P69FQC,2180-03-23,2180-03-23 11:51:00,70017,SEROLOGY/BLOOD,1,...,None,<NA>,None,<NA>,None,None,None,NaN,None,NONREACTIVE. Reference Range: Non-Reactive.
3,4,10000032,<NA>,5401234,P69FQC,2180-03-23,2180-03-23 11:51:00,70017,SEROLOGY/BLOOD,1,...,None,<NA>,None,<NA>,None,None,None,NaN,None,POSITIVE BY EIA. A positive IgG result genera...
4,5,10000032,<NA>,6287540,P69FQC,2180-03-23,2180-03-23 11:51:00,70046,IMMUNOLOGY,1,...,None,<NA>,None,<NA>,None,None,None,NaN,None,HIV-1 RNA is not detected. Performed using th...


#### hosp/labevents (lab results)

In [19]:
chunk.head()

,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,1,10000032,<NA>,2704548,50931,P69FQC,2180-03-23 11:51:00,2180-03-23 15:56:00,___,95.0,mg/dL,70.0,100.0,None,ROUTINE,"IF FASTING, 70-100 NORMAL, >125 PROVISIONAL DI..."
1,2,10000032,<NA>,36092842,51071,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,None,NaN,NaN,None,ROUTINE,None
2,3,10000032,<NA>,36092842,51074,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,None,NaN,NaN,None,ROUTINE,None
3,4,10000032,<NA>,36092842,51075,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,None,NaN,NaN,None,ROUTINE,BENZODIAZEPINE IMMUNOASSAY SCREEN DOES NOT DET...
4,5,10000032,<NA>,36092842,51079,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,None,NaN,NaN,None,ROUTINE,None


#### hosp/hcpcsevents

In [17]:
chunk.head()

,subject_id,hadm_id,chartdate,hcpcs_cd,seq_num,short_description
0,10000068,25022803,2160-03-04,99218,1,Hospital observation services
1,10000084,29888819,2160-12-28,G0378,1,Hospital observation per hr
2,10000108,27250926,2163-09-27,99219,1,Hospital observation services
3,10000117,22927623,2181-11-15,43239,1,Digestive system
4,10000117,22927623,2181-11-15,G0378,2,Hospital observation per hr


#### hosp/emar_detail

In [13]:
chunk.head()

,subject_id,emar_id,emar_seq,parent_field_ordinal,administration_type,pharmacy_id,barcode_type,reason_for_no_barcode,complete_dose_not_given,dose_due,...,infusion_rate_unit,route,infusion_complete,completion_interval,new_iv_bag_hung,continued_infusion_in_other_location,restart_interval,side,site,non_formulary_visual_verification
0,10000032,10000032-10,10,1.1,None,86768272,if,None,<NA>,None,...,None,None,None,None,None,None,None,None,None,None
1,10000032,10000032-10,10,NaN,Standard Maintenance Medication,<NA>,None,None,False,5000,...,None,None,None,None,None,None,None,None,None,None
2,10000032,10000032-100,100,1.1,None,37992677,if,None,<NA>,None,...,None,None,None,None,None,None,None,None,None,None
3,10000032,10000032-100,100,NaN,Standard Maintenance Medication,<NA>,None,None,False,5000,...,None,None,None,None,None,None,None,None,None,None
4,10000032,10000032-101,101,1.1,None,60826865,if,None,<NA>,None,...,None,None,None,None,None,None,None,None,None,None


#### hosp/emar

In [9]:
chunk.head()

,subject_id,hadm_id,emar_id,emar_seq,poe_id,pharmacy_id,enter_provider_id,charttime,medication,event_txt,scheduletime,storetime
0,10000032,22595853,10000032-10,10,10000032-32,86768272,None,2180-05-07 07:51:00,Heparin,Administered,2180-05-07 08:00:00,2180-05-07 07:56:00
1,10000032,22595853,10000032-11,11,10000032-37,43092008,None,2180-05-07 07:51:00,Raltegravir,Administered,2180-05-07 08:00:00,2180-05-07 07:56:00
2,10000032,22595853,10000032-12,12,10000032-27,30740602,None,2180-05-07 07:51:00,Sodium Chloride 0.9% Flush,Flushed,2180-05-07 07:51:00,2180-05-07 07:56:00
3,10000032,22595853,10000032-13,13,10000032-36,<NA>,None,2180-05-07 08:00:00,Nicotine Patch,Not Applied,2180-05-07 08:00:00,2180-05-07 07:57:00
4,10000032,22595853,10000032-14,14,10000032-35,<NA>,None,2180-05-07 08:00:00,Furosemide,Not Given,2180-05-07 08:00:00,2180-05-07 09:27:00


#### hosp/drgcodes

In [7]:
chunk.head()

,subject_id,hadm_id,drg_type,drg_code,description,drg_severity,drg_mortality
0,10000032,22595853,APR,283,OTHER DISORDERS OF THE LIVER,2,2
1,10000032,22595853,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA ...",<NA>,<NA>
2,10000032,22841357,APR,279,HEPATIC COMA AND OTHER MAJOR ACUTE LIVER DISOR...,3,2
3,10000032,22841357,HCFA,442,"DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA ...",<NA>,<NA>
4,10000032,25742920,APR,283,OTHER DISORDERS OF THE LIVER,3,2


#### hosp/diagnoses_icd (diagnosis code for each admission)

In [41]:
chunk

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
...,...,...,...,...,...
2038084,13196233,28822279,12,2449,9
2038085,13196233,28822279,13,7821,9
2038086,13196233,28822279,14,52809,9
2038087,13196233,28822279,15,E9331,9


#### hosp/d_labitems

In [39]:
chunk.head()

,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


#### hosp/d_icd_procedures

In [35]:
chunk.head()

,icd_code,icd_version,long_title
0,0001,9,Therapeutic ultrasound of vessels of head and ...
1,0002,9,Therapeutic ultrasound of heart
2,0003,9,Therapeutic ultrasound of peripheral vascular ...
3,0009,9,Other therapeutic ultrasound
4,001,10,"Central Nervous System and Cranial Nerves, Bypass"


#### hosp/d_icd_diagnoses

In [33]:
chunk.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


#### hosp/d_hcpcs

In [31]:
chunk.head()

,code,category,long_description,short_description
0,00000,<NA>,None,Invalid Code
1,0001F,2,None,Composite measures
2,0002F,2,None,Composite measures
3,0003F,2,None,Composite measures
4,0004F,2,None,Composite measures


#### hosp/admission (admissions/ discharge time)

In [29]:
chunk.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaT,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaT,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaT,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaT,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaT,EU OBSERVATION,P39NWO,EMERGENCY ROOM,None,None,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


#### hosp/patients (Demographic + death info)

In [4]:
df['dead'] = df['dod'].isna()

In [7]:
df['dead'].value_counts()

dead
True     326326
False     38301
Name: count, dtype: int64

In [ ]:
# Patients

df

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaT
2,10000058,F,33,2168,2020 - 2022,NaT
3,10000068,F,19,2160,2008 - 2010,NaT
4,10000084,M,72,2160,2017 - 2019,2161-02-13
...,...,...,...,...,...,...
364622,19999828,F,46,2147,2017 - 2019,NaT
364623,19999829,F,28,2186,2008 - 2010,NaT
364624,19999840,M,58,2164,2008 - 2010,2164-09-17
364625,19999914,F,49,2158,2017 - 2019,NaT


#### hosp/transfer

In [5]:
#Transfer

df

,subject_id,hadm_id,transfer_id,eventtype,careunit,intime,outtime
0,10000032,22595853,33258284,ED,Emergency Department,2180-05-06 19:17:00,2180-05-06 23:30:00
1,10000032,22595853,35223874,admit,Transplant,2180-05-06 23:30:00,2180-05-07 17:21:27
2,10000032,22595853,36904543,discharge,UNKNOWN,2180-05-07 17:21:27,NaT
3,10000032,22841357,34100253,discharge,UNKNOWN,2180-06-27 18:49:12,NaT
4,10000032,22841357,34703856,admit,Transplant,2180-06-26 21:31:00,2180-06-27 18:49:12
...,...,...,...,...,...,...,...
2413576,19999914,<NA>,32002659,ED,Emergency Department,2158-12-24 11:41:00,2158-12-24 11:56:00
2413577,19999987,23865745,30249304,transfer,Neurology,2145-11-04 21:29:30,2145-11-11 13:00:47
2413578,19999987,23865745,34731548,ED,Emergency Department,2145-11-02 19:28:00,2145-11-02 22:59:00
2413579,19999987,23865745,36195440,admit,Trauma SICU (TSICU),2145-11-02 22:59:00,2145-11-04 21:29:30


In [5]:
df.shape

(20292611, 21)

In [ ]:
#prescription

df.head(2)

,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,...,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10000032,22595853,12775705,10000032-55,55,P85UQ1,2180-05-08 08:00:00,2180-05-07 22:00:00,MAIN,Furosemide,...,008209,51079007320,40mg Tablet,None,40,mg,1,TAB,1,PO/NG
1,10000032,22595853,18415984,10000032-42,42,P23SJA,2180-05-07 02:00:00,2180-05-07 22:00:00,MAIN,Ipratropium Bromide Neb,...,021700,00487980125,2.5mL Vial,None,1,NEB,1,VIAL,4,IH


In [ ]:
#diagnosis_icd

df

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
...,...,...,...,...,...
6364483,19999987,23865745,7,41401,9
6364484,19999987,23865745,8,78039,9
6364485,19999987,23865745,9,0413,9
6364486,19999987,23865745,10,36846,9


In [13]:
df.head()

,subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,amount,amountuom,...,ordercomponenttypedescription,ordercategorydescription,patientweight,totalamount,totalamountuom,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
0,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:01:00,2180-07-23 18:56:00,226452,200.000000,mL,...,Main order parameter,Bolus,39.4,200.0,mL,0,0,FinishedRunning,200.0,200.0
1,10000032,29079034,39553978,18704,2180-07-23 17:00:00,2180-07-23 17:30:00,2180-07-23 17:02:00,220862,49.999999,mL,...,Main order parameter,Continuous IV,39.4,50.0,mL,0,0,FinishedRunning,50.0,100.0
2,10000032,29079034,39553978,18704,2180-07-23 17:33:00,2180-07-23 18:03:00,2180-07-23 18:16:00,220862,49.999999,mL,...,Main order parameter,Continuous IV,39.4,50.0,mL,0,0,FinishedRunning,50.0,100.0
3,10000032,29079034,39553978,18704,2180-07-23 18:56:00,2180-07-23 18:57:00,2180-07-23 18:56:00,226452,100.000000,mL,...,Main order parameter,Bolus,39.4,100.0,mL,0,0,FinishedRunning,100.0,100.0
4,10000032,29079034,39553978,20925,2180-07-23 21:10:00,2180-07-23 21:11:00,2180-07-23 21:10:00,226452,100.000000,mL,...,Main order parameter,Bolus,39.4,100.0,mL,0,0,FinishedRunning,100.0,100.0
